# Phase 1 — Final 40k BPE Tokenizer

This Kaggle notebook trains a **40,000-token BPE vocabulary** on the four official Phase-1 training files from `Exploration-Lab/CS-781-Capstone`.

It is streaming-oriented: it does not concatenate the whole corpus into one giant Python string. It:
1. verifies the Hugging Face repository and all four files,
2. downloads the files,
3. cleans the legal text,
4. sanity-checks parsing and cleaning,
5. trains a 40k BPE tokenizer on the full cleaned corpus,
6. sanity-checks encode/decode and special tokens,
7. saves and reloads the tokenizer,
8. counts the resulting BPE tokens in the complete corpus,
9. checks whether the proposed 60/20/15/5 mixture is feasible without oversampling.


In [1]:
# ============================================================
# CELL 1 — INSTALL / IMPORTS
# ============================================================
!pip -q install tokenizers huggingface_hub ijson tqdm pandas

import os, re, json, html, unicodedata, time, gc
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 2.5 MB/s eta 0:00:00a 0:00:01


In [2]:
# ============================================================
# CELL 2 — EXACT DATASET CONFIGURATION
# ============================================================
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

HF_FILES = {
    "Acts": "Phase_1_Pretraining/Train/train_acts.json",
    "High Courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "Supreme Court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

VOCAB_SIZE = 40_000
MIN_CHARS = 50

OUTPUT_DIR = Path("/kaggle/working/phase1_40k_bpe")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", HF_DATASET_REPO)
print("Target BPE vocabulary:", f"{VOCAB_SIZE:,}")
print("Output directory:", OUTPUT_DIR)


Dataset: Exploration-Lab/CS-781-Capstone
Target BPE vocabulary: 40,000
Output directory: /kaggle/working/phase1_40k_bpe


In [3]:
# ============================================================
# CELL 3 — HF TOKEN + REPOSITORY SANITY CHECK
# ============================================================
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found. Add a Kaggle secret named HF_TOKEN."

api = HfApi(token=HF_TOKEN)

repo_files = api.list_repo_files(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
)

print("Repository accessible:", HF_DATASET_REPO)
print("Repository file count:", len(repo_files))

missing = [f for f in HF_FILES.values() if f not in repo_files]

if missing:
    raise FileNotFoundError(
        "Missing required Phase-1 files:\n" + "\n".join(missing)
    )

print("\nAll four Phase-1 training files exist:")
for name, filename in HF_FILES.items():
    print(f"  [OK] {name:20s} -> {filename}")

print("\nREPOSITORY SANITY CHECK PASSED")


Repository accessible: Exploration-Lab/CS-781-Capstone
Repository file count: 34

All four Phase-1 training files exist:
  [OK] Acts                 -> Phase_1_Pretraining/Train/train_acts.json
  [OK] High Courts          -> Phase_1_Pretraining/Train/train_high_courts.json
  [OK] Supreme Court        -> Phase_1_Pretraining/Train/train_supreme_court.json
  [OK] Tribunals            -> Phase_1_Pretraining/Train/train_tribunals.json

REPOSITORY SANITY CHECK PASSED


In [4]:
# ============================================================
# CELL 4 — DOWNLOAD THE FOUR FILES
# ============================================================
LOCAL_FILES = {}

for source, filename in HF_FILES.items():
    print(f"Downloading {source}...")

    LOCAL_FILES[source] = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    print(" ", LOCAL_FILES[source])

print("\nDownload sanity check:")

for source, path in LOCAL_FILES.items():
    size_gb = Path(path).stat().st_size / (1024 ** 3)
    print(f"  {source:20s}: {size_gb:.2f} GB")


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json


Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json

Download sanity check:
  Acts                : 0.68 GB
  High Courts         : 9.94 GB
  Supreme Court       : 1.23 GB
  Tribunals           : 6.04 GB


## Cleaning policy

The cleaner is intentionally conservative.

It removes obvious:
- HTML/XML markup
- URLs
- HTML entities
- Word/XML control artifacts
- replacement characters
- zero-width formatting artifacts
- excessive whitespace

It **does not remove normal legal words** such as `judgment`, `order`, `petitioner`, etc. Those are legal language and should remain in the corpus.


In [5]:
# ============================================================
# CELL 5 — CLEANING + TEXT EXTRACTION
# ============================================================
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.I)
TAG_RE = re.compile(r"<[^>]+>")
HTML_ENTITY_RE = re.compile(
    r"&(?:nbsp|amp|lt|gt|quot|apos|#\d+|#x[0-9a-fA-F]+);",
    re.I
)
CONTROL_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
WEIRD_SPACE_RE = re.compile(
    r"[\u00a0\u1680\u180e\u2000-\u200a\u202f\u205f\u3000]"
)

def clean_text(text):
    if text is None:
        return ""

    if isinstance(text, list):
        text = " ".join(str(x) for x in text)
    elif isinstance(text, dict):
        text = " ".join(str(v) for v in text.values())
    else:
        text = str(text)

    text = html.unescape(text)
    text = TAG_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = HTML_ENTITY_RE.sub(" ", text)

    text = text.replace("\ufeff", " ")
    text = text.replace("\ufffd", " ")

    text = CONTROL_RE.sub(" ", text)
    text = WEIRD_SPACE_RE.sub(" ", text)

    text = unicodedata.normalize("NFKC", text)

    # Remove zero-width formatting characters.
    text = re.sub(r"[\u200b\u200c\u200d\u2060]", "", text)

    # Normalize whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text


def extract_text(record):
    if isinstance(record, str):
        return record

    if isinstance(record, dict):
        # Prefer likely legal-text fields.
        for key in ("text", "content", "judgment", "document", "body"):
            if key in record:
                value = record[key]

                if isinstance(value, list):
                    return " ".join(map(str, value))

                if isinstance(value, str):
                    return value

        # Fallback: collect string/list values.
        parts = []

        for value in record.values():
            if isinstance(value, str):
                parts.append(value)
            elif isinstance(value, list):
                parts.extend(
                    str(x)
                    for x in value
                    if isinstance(x, (str, int, float))
                )

        return " ".join(parts)

    return str(record)


In [6]:
# ============================================================
# CELL 6 — STREAMING JSON PARSER + SANITY TEST
# ============================================================
import ijson

def iter_json_records(path):
    # Detect top-level JSON structure without loading the file.
    with open(path, "rb") as f:
        first = f.read(4096).lstrip()[:1]

    if first == b"[":
        with open(path, "rb") as f:
            for item in ijson.items(f, "item"):
                yield item
        return

    if first == b"{":
        with open(path, "rb") as f:
            for _, value in ijson.kvitems(f, ""):
                yield value
        return

    raise ValueError(f"Unsupported JSON structure: {path}")


print("Testing one record from every source...\n")

for source, path in LOCAL_FILES.items():
    first_record = next(iter_json_records(path))
    text = clean_text(extract_text(first_record))

    print(f"{source}: {len(text):,} cleaned characters")
    print("  sample:", repr(text[:250]))
    print()

    assert len(text) > 0, f"Empty first record for {source}"

print("PARSING + CLEANING SANITY CHECK PASSED")


Testing one record from every source...

Acts: 7,783 cleaned characters
  sample: 'Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 19'

High Courts: 1,030 cleaned characters
  sample: "Court No. - 24 Case :- SECOND APPEAL No. - 550 of 1986 Appellant :- Ram Naresh Singh Baghel Respondent :- Nagar Maha Palika Lko. Counsel for Appellant :- K.C.Jauhari,Manish Jauhari,Mohit Jauhari,Raj Kumar Srivastava,Ram Bali Ram,V.P. Mishra Hon'ble M"

Supreme Court: 24,250 cleaned characters
  sample: 'PETITIONER: HIS HOLINESS SRI VISHWOTHAMA THIRTHA SWAMIAR OFSODE MUTT Vs. RESPONDENT: THE STATE OF MYSORE DATE OF JUDGMENT: 18/12/1961 ACT: Temple-Large number of pilgrims visiting and worshiping-If private or public-Madras Temple Entry Authorisation '

Tribunals: 12,312 cleaned characters
  sample: 'Appellate

In [7]:
# ============================================================
# CELL 7 — STREAMING BPE CORPUS ITERATOR
# ============================================================
def corpus_iterator():
    # IMPORTANT:
    # This iterates over the COMPLETE cleaned corpus.
    # Nothing is concatenated into one huge in-memory string.

    for source, path in LOCAL_FILES.items():
        print(f"Starting corpus stream: {source}")

        for record in iter_json_records(path):
            text = clean_text(extract_text(record))

            if len(text) >= MIN_CHARS:
                yield text


# Preview only; this does not store the corpus.
preview = corpus_iterator()

for i in range(3):
    sample = next(preview)
    print(f"Sample {i + 1}:")
    print(sample[:400])
    print("\n" + "-" * 80 + "\n")

print("STREAMING ITERATOR SANITY CHECK PASSED")


Starting corpus stream: Acts
Sample 1:
Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors including ground floor. The Corporation of the City of Bangalore granted permission for construction of b

--------------------------------------------------------------------------------

Sample 2:
The Bangalore Palace (Acquisition And Transfer) Act, 1996 [18th November, 1996] Karnataka Act No. 18 of 1996 An Act to provide for the acquisition and transfer of the Bangalore Palace and open space around it in the public interest and for its preservation and for matters connected therewith. WHEREAS, the Palace at Bangalore popularly known as the Bangalore Palace, Karnataka's unique historical an

-----------------------------------------------------------------

In [8]:
# ============================================================
# CELL 8 — TRAIN THE 40K BPE TOKENIZER ON THE FULL CORPUS
# ============================================================
from tokenizers import Tokenizer, models, trainers
from tokenizers import pre_tokenizers, decoders, normalizers

tokenizer = Tokenizer(
    models.BPE(unk_token="<unk>")
)

tokenizer.normalizer = normalizers.NFKC()

# Byte-level BPE handles arbitrary Unicode/UTF-8 text robustly.
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(
    add_prefix_space=False
)
tokenizer.decoder = decoders.ByteLevel()

SPECIAL_TOKENS = [
    "<pad>",
    "<unk>",
    "<bos>",
    "<eos>",
]

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS,
    show_progress=True,
)

print(
    f"Training a {VOCAB_SIZE:,}-vocabulary BPE "
    "on the COMPLETE cleaned corpus..."
)

start_time = time.time()

tokenizer.train_from_iterator(
    corpus_iterator(),
    trainer=trainer,
    length=None,
)

elapsed = time.time() - start_time

print("\nBPE TRAINING COMPLETE")
print("Actual vocabulary size:", tokenizer.get_vocab_size())
print("Elapsed time:", f"{elapsed / 3600:.2f} hours")

assert tokenizer.get_vocab_size() <= VOCAB_SIZE


Training a 40,000-vocabulary BPE on the COMPLETE cleaned corpus...
Starting corpus stream: Acts
Starting corpus stream: High Courts
Starting corpus stream: Supreme Court
Starting corpus stream: Tribunals




BPE TRAINING COMPLETE
Actual vocabulary size: 40000
Elapsed time: 1.97 hours


In [9]:
# ============================================================
# CELL 9 — TOKENIZER SANITY TEST
# ============================================================
samples = [
    "The Supreme Court held that the appellant was entitled to relief.",
    "Section 302 of the Indian Penal Code was considered by the Court.",
    "The petitioner challenged the judgment and order of the High Court.",
    "Article 21 of the Constitution of India protects the right to life.",
]

print("Vocabulary size:", tokenizer.get_vocab_size())

print("\nSpecial-token IDs:")

for token in SPECIAL_TOKENS:
    token_id = tokenizer.token_to_id(token)

    print(f"  {token:6s} -> {token_id}")

    assert token_id is not None

print("\nEncode/decode tests:")

for text in samples:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids)

    print("\nINPUT:")
    print(text)

    print("\nTOKEN COUNT:")
    print(len(encoded.ids))

    print("\nFIRST TOKENS:")
    print(encoded.tokens[:30])

    print("\nDECODED:")
    print(decoded)

    print("\n" + "-" * 80)

    assert len(encoded.ids) > 0

print("\nTOKENIZER SANITY TEST PASSED")


Vocabulary size: 40000

Special-token IDs:
  <pad>  -> 0
  <unk>  -> 1
  <bos>  -> 2
  <eos>  -> 3

Encode/decode tests:

INPUT:
The Supreme Court held that the appellant was entitled to relief.

TOKEN COUNT:
12

FIRST TOKENS:
['The', 'ĠSupreme', 'ĠCourt', 'Ġheld', 'Ġthat', 'Ġthe', 'Ġappellant', 'Ġwas', 'Ġentitled', 'Ġto', 'Ġrelief', '.']

DECODED:
The Supreme Court held that the appellant was entitled to relief.

--------------------------------------------------------------------------------

INPUT:
Section 302 of the Indian Penal Code was considered by the Court.

TOKEN COUNT:
13

FIRST TOKENS:
['Section', 'Ġ302', 'Ġof', 'Ġthe', 'ĠIndian', 'ĠPenal', 'ĠCode', 'Ġwas', 'Ġconsidered', 'Ġby', 'Ġthe', 'ĠCourt', '.']

DECODED:
Section 302 of the Indian Penal Code was considered by the Court.

--------------------------------------------------------------------------------

INPUT:
The petitioner challenged the judgment and order of the High Court.

TOKEN COUNT:
12

FIRST TOKENS:
['The', 'Ġp

In [10]:
# ============================================================
# CELL 10 — SAVE + RELOAD THE FINAL 40K TOKENIZER
# ============================================================
TOKENIZER_PATH = OUTPUT_DIR / "phase1_bpe_40k.json"

tokenizer.save(str(TOKENIZER_PATH))

print("Saved tokenizer:")
print(TOKENIZER_PATH)

print(
    "Tokenizer file size:",
    f"{TOKENIZER_PATH.stat().st_size / (1024 ** 2):.2f} MB"
)

# Reload sanity check.
reloaded = Tokenizer.from_file(str(TOKENIZER_PATH))

assert reloaded.get_vocab_size() == tokenizer.get_vocab_size()

test_text = samples[0]

assert (
    reloaded.encode(test_text).ids
    == tokenizer.encode(test_text).ids
)

print("\nSAVE + RELOAD SANITY CHECK PASSED")


Saved tokenizer:
/kaggle/working/phase1_40k_bpe/phase1_bpe_40k.json
Tokenizer file size: 2.69 MB

SAVE + RELOAD SANITY CHECK PASSED


In [ ]:
# ============================================================
# CELL 11 — COUNT BPE TOKENS IN THE COMPLETE CORPUS
# ============================================================
# This is separate from tokenizer training.
#
# Now that the tokenizer is frozen, we measure exactly how many
# BPE token OCCURRENCES each source contains.

def count_source_tokens(path, tokenizer):
    documents = 0
    characters = 0
    whitespace_words = 0
    bpe_tokens = 0

    for record in iter_json_records(path):
        text = clean_text(extract_text(record))

        if len(text) < MIN_CHARS:
            continue

        documents += 1
        characters += len(text)
        whitespace_words += len(text.split())

        bpe_tokens += len(tokenizer.encode(text).ids)

    return (
        documents,
        characters,
        whitespace_words,
        bpe_tokens,
    )


rows = []

start_time = time.time()

for source, path in LOCAL_FILES.items():

    print(f"Counting BPE tokens: {source}")

    documents, characters, words, tokens = count_source_tokens(
        path,
        tokenizer
    )

    rows.append({
        "source": source,
        "documents": documents,
        "characters": characters,
        "whitespace_words": words,
        "bpe_tokens": tokens,
        "tokens_per_word": (
            tokens / words if words else 0
        ),
        "chars_per_token": (
            characters / tokens if tokens else 0
        ),
    })


counts_df = pd.DataFrame(rows)

total_tokens = counts_df["bpe_tokens"].sum()

counts_df["token_share"] = (
    counts_df["bpe_tokens"] / total_tokens
)

print("\nCOMPLETE CORPUS TOKEN COUNTS")
display(counts_df)

print(
    "\nTOTAL BPE TOKEN OCCURRENCES:",
    f"{total_tokens:,}"
)

print(
    "TOTAL BPE TOKENS:",
    f"{total_tokens / 1e9:.3f} billion"
)

print(
    "Counting time:",
    f"{(time.time() - start_time) / 3600:.2f} hours"
)

counts_df.to_csv(
    OUTPUT_DIR / "complete_corpus_40k_token_counts.csv",
    index=False
)


Counting BPE tokens: Acts
Counting BPE tokens: High Courts
Counting BPE tokens: Supreme Court
Counting BPE tokens: Tribunals


In [ ]:
# ============================================================
# CELL 12 — 60/20/15/5 TOKEN-MIXTURE FEASIBILITY
# ============================================================
TARGET_MIX = {
    "Supreme Court": 0.60,
    "High Courts": 0.20,
    "Tribunals": 0.15,
    "Acts": 0.05,
}

available = dict(
    zip(
        counts_df["source"],
        counts_df["bpe_tokens"]
    )
)

# Largest total budget that can satisfy the mixture
# WITHOUT oversampling any source.
max_total_without_oversampling = min(
    available[source] / share
    for source, share in TARGET_MIX.items()
)

print("Available BPE tokens:")

for source, tokens in available.items():
    print(f"  {source:20s}: {tokens:,}")

print(
    "\nMaximum total training budget that can satisfy "
    "60/20/15/5 without oversampling:"
)

print(
    f"  {max_total_without_oversampling:,.0f} tokens"
)

allocation_rows = []

for source, share in TARGET_MIX.items():

    target = max_total_without_oversampling * share

    allocation_rows.append({
        "source": source,
        "target_share": share,
        "target_tokens": int(target),
        "available_tokens": int(available[source]),
        "enough_without_oversampling": (
            available[source] >= target
        ),
    })

allocation_df = pd.DataFrame(allocation_rows)

display(allocation_df)

allocation_df.to_csv(
    OUTPUT_DIR / "60_20_15_5_feasibility.csv",
    index=False
)


In [ ]:
# ============================================================
# CELL 13 — FINAL SANITY CHECK
# ============================================================
assert TOKENIZER_PATH.exists()

assert tokenizer.get_vocab_size() <= 40_000

assert total_tokens > 0

assert set(counts_df["source"]) == set(
    HF_FILES.keys()
)

assert abs(
    counts_df["token_share"].sum() - 1.0
) < 1e-6

for token in SPECIAL_TOKENS:
    assert tokenizer.token_to_id(token) is not None

print("==============================================")
print("FINAL PHASE-1 BPE SANITY CHECK PASSED")
print("==============================================")
print(
    "Vocabulary size:",
    f"{tokenizer.get_vocab_size():,}"
)
print(
    "Complete cleaned corpus:",
    f"{total_tokens:,} BPE token occurrences"
)
print(
    "Complete corpus:",
    f"{total_tokens / 1e9:.3f} billion BPE tokens"
)
print(
    "Tokenizer:",
    TOKENIZER_PATH
)

print("\nThe tokenizer is now frozen.")
print(
    "Next step: use this exact tokenizer to construct "
    "the final Phase-1 training-token mixture."
)
